# Alice & Bob Resource Estimator Tutorial

This Jupyter notebook is a tutorial for the **Alice & Bob Resource Estimator**, a tool designed to estimate the physical resources required to execute quantum algorithms on Alice & Bob hardware.

The tutorial introduces the estimator’s concepts, workflow, and usage through concrete examples, with a focus on elliptic curve cryptography (ECC).

# Table of Contents
1. [Prerequisites](#prerequisites)
2. [What Is the Alice & Bob Resource Estimator?](#what-is-the-alice--bob-resource-estimator)
3. [Installation](#installation)
4. [Examples](#examples)
   1. [ECC Based on arXiv:2302.06639](#ecc-based-on-arxiv230206639)
   2. [ECC Based on Logical Counts](#ecc-based-on-logical-counts)
   3. [ECC Based on Qualtran Implementation](#ecc-based-on-qualtran-implementation)
   4. [Estimation from a Q# File](#estimation-from-a-q-file)


## Prerequisites

The Alice & Bob Resource Estimator provides a Python interface, and no knowledge of Rust is required. However, readers are expected to have the following background:

- Familiarity with basic Python syntax
- A basic understanding of quantum circuits and common quantum gates
- A basic understanding of cat qubits
- Some familiarity with quantum error correction (QEC)

While **prior knowledge of elliptic curve cryptography** is helpful for interpreting the example results, it is **not required** to use the resource estimator itself.


## What Is the Alice & Bob Resource Estimator?

The [Microsoft Azure Quantum Resource Estimator (AQRE)](https://arxiv.org/abs/2311.05801) introduces a layered framework that connects high-level quantum programs to physical resource estimates through an intermediate quantum instruction set architecture (ISA).

The **Alice & Bob Resource Estimator** builds on this framework by specializing the hardware and error-correction layers for Alice & Bob’s **cat-qubit architecture**. Given a quantum algorithm expressed in a high-level programming language, the estimator:

1. Compiles the program to a quantum ISA,
2. Applies architecture-specific overhead models, and
3. Produces detailed physical resource estimates.

These overhead models account for factors such as routing, quantum error correction, and magic-state factory implementations.


## Installation

This repository includes a `pixi.toml` file to enable reproducible environments.

To install the required environment and the Python extension, run the following commands:

To run the setup commands, you’ll need Pixi installed on your machine. Pixi will create and manage the project environment for you, including installing Python, Rust, and any required dependencies defined in the project configuration. In addition, you’ll need a working native build toolchain for your operating system (Xcode Command Line Tools on macOS, build-essential on Linux, or Microsoft C++ Build Tools on Windows), since maturin develop compiles Rust extensions locally.


In [1]:
!pixi install
!pixi run maturin develop --uv

⠁ updating pypi packages in 'default'                                           ✔ The default environment has been installed.
⠁ activating environment                                                        🔗 Found pyo3 bindings
🐍 Found CPython 3.13 at /Users/ngialour/Desktop/Rust/qsharp-alice-bob-resource-estimator/.pixi/envs/default/bin/python
📡 Using build options features from pyproject.toml
   Compiling pyo3-build-config v0.25.1
   Compiling pyo3-ffi v0.25.1=======>  ] 153/165: pyo3-build-config           m    Building [======================>  ] 152/165: pyo3-build-config(build)    
   Compiling pyo3-macros-backend v0.25.1
   Compiling pyo3 v0.25.1
   Compiling pyo3-macros v0.25.1=====> ] 161/165: pyo3-macros-backend         
   Compiling qsharp-alice-bob-resource-estimator v0.1.0 (/Users/ngialour/Desktop/Rust/qsharp-alice-bob-resource-estimator)
    Finished `dev` profile [unoptimized + debuginfo] target(s) in 4.32surce...
📦 Built wheel for CPython 3.13 to /var/folders/4d/ryg0k1h

Now you can import the resource estimator from Rust just like any other Python module.

In [2]:
# Import the Rust functionality as a Python module
import qsharp_alice_bob_resource_estimator as qre

# Examples
## ECC based on arXiv:2302.06639
The resource estimator includes a built-in example for computing elliptic curve discrete logarithms, based on the methodology described in [arXiv:2302.06639](https://arxiv.org/abs/2302.06639). The relevant function is documented below.

In [3]:
print(qre.elliptic_curve_estimate_struct.__doc__)  # type: ignore

Structured variant of the ECC example that returns typed estimate objects.

# Arguments
- `bit_size` — ECC modulus bit size.
- `window_size` — Window size used by the example algorithm.
- `frontier` — If `true`, also return a list representing the frontier.

# Returns
A tuple:
1. `EstimatesPy` — single best estimate,
2. `Vec<EstimatesPy>` — optional frontier (empty if `frontier == false`).

# Errors
Propagates example execution or estimation errors as Python `RuntimeError`s.


To run the example, it suffices to specify a bit size and a window size, which is used to perform windowed arithmetic.

In [4]:
elliptic_curve, _ = qre.elliptic_curve_estimate_struct(256, 18, True)  # type: ignore
print("\n=== Elliptic curve cryptography example ===")
print(elliptic_curve)


=== Elliptic curve cryptography example ===
Parameters obtained from the Rust resource estimator
─────────────────────────────
# physical qubits:    115,203
runtime:             7.57 hrs
total error:         0.23104
─────────────────────────────
code distance:       13 (|α|² = 19.00)
#factories:          61
factories distance:  9 (|α|² = 12.83)
factory fraction:    4.50%
─────────────────────────────



The output summarizes the physical resource estimates produced by the Alice & Bob Resource Estimator for the elliptic curve cryptography example. It reports the total number of physical qubits required, the estimated runtime, and the resulting total logical error. Additional details describe the chosen quantum error-correcting code parameters, including the code distance and corresponding cat-state size, as well as the configuration of magic-state factories. In particular, the number of factories, their code distance, and the fraction of resources they consume provide insight into the overhead associated with fault-tolerant gate synthesis in this setting.

The parameters can be easily accessed as class variables as seen below 

In [5]:
code_distance = elliptic_curve.code_distance
print(f"Code distance: {code_distance}")
# Same for other parameters

Code distance: 13


## ECC based on logical counts

The same example can be executed in an equivalent manner using the more general interface, which accepts the tuple ``(#Qubits, #CX, #CCX)`` as input. The corresponding function docstring is shown below.

In [6]:
print(qre.estimate_resources_struct.__doc__)  # type: ignore

Estimate resources from explicit logical counts and return typed results,
optionally including a frontier of trade-offs.

# Arguments
- `qubits` — Logical (algorithm) qubit count.
- `cx` — Logical CX-equivalent two-qubit gate count.
- `ccx` — Logical CCX (Toffoli) gate count.
- `frontier` — If `true`, compute and return the frontier as structured objects.
- `error_total` — Overall error target; mutually exclusive with `error_budget`.
- `error_budget` — Tuple `(target, meas, routing)` for an explicit split.

# Returns
A tuple:
1. `EstimatesPy` — single best estimate,
2. `Vec<EstimatesPy>` — frontier (empty if `frontier == false`).

# Errors
Propagates errors from the physical resource estimator.


To obtain the logical resource counts, we reproduce the calculations described in [arXiv:2302.06639](https://arxiv.org/abs/2302.06639). As these calculations are identical to those used in the previous example, the resulting estimates are expected to match exactly. The only difference is in how the logical quantities are provided to the resource estimator: in the previous example, they are computed internally from the specified bit size and window size, whereas here the same logical counts are computed a priori and passed directly as input.

In [7]:
import math

bit_size = 256
window_size = 18

qubit_count = 9 * bit_size + window_size + 4

# Asymptotic gate counts, arXiv:2302.06639 (p. 21, app C.10)
cx_count = math.ceil(448 * bit_size**3 / window_size)
ccx_count = math.ceil(348 * bit_size**3 / window_size)

elliptic_curve_logic, _ = qre.estimate_resources_struct(  # type: ignore
    qubit_count,
    cx_count,
    ccx_count,
    frontier=False,
    error_total=None,
    error_budget=(0.333 * 0.5, 0.333 * 0.5, 0.0),
)

In [8]:
print("\n=== Logical_estimates example ===")
print(elliptic_curve_logic)


=== Logical_estimates example ===
Parameters obtained from the Rust resource estimator
─────────────────────────────
# physical qubits:    115,203
runtime:             7.57 hrs
total error:         0.23104
─────────────────────────────
code distance:       13 (|α|² = 19.00)
#factories:          61
factories distance:  9 (|α|² = 12.83)
factory fraction:    4.50%
─────────────────────────────



## ECC based on Qualtran implementation
A third option is to use [Qualtran](https://qualtran.readthedocs.io/en/latest/index.html). Leveraging our provided primitives, the algorithm described in [arXiv:2302.06639](https://arxiv.org/abs/2302.06639) can be constructed within the Qualtran framework. Qualtran then produces an estimate of the logical resource counts, which can subsequently be used as input to the resource estimator.

In [9]:
# Import Qualtran primitives
from ecc_primitives.construction_helpers import (
    ECCInstance,
    create_ecc_circuit,
)
from qualtran_helpers.qualtran_wrapper import estimate_from_qualtran

In [10]:
# Initialize configuration for secp256k1 curve
cfg = ECCInstance(
    n=256,  # bitsize
    p=2**256 - 2**32 - 977,  # prime
    wa=18,  # window size for elliptic curve multiplication
    wm=6,  # window size for the integer multiplication
    Gx=55066263022277343669578718895168534326250603453777594175500187360389116729240,  # x coordinate of the generator
    Gy=32670510020758816978083085130507043184471273380659243275938904335757337482424,  # y coordinate of the generator
    scalar=2026,  # private key to find
)

ecc_circuit = create_ecc_circuit(cfg)

The resource estimation can then be run by using the function ``estimate_from_qualtran`` as shown below

In [11]:
ecc_result = estimate_from_qualtran(
    ecc_circuit,
    frontier=False,
    error_total=None,
    error_budget=(0.333 * 0.5, 0.333 * 0.5, 0.0),
)

In [12]:
print("\n=== Qualtran example ===")
print(ecc_result)


=== Qualtran example ===
Parameters obtained from the Rust resource estimator
─────────────────────────────
# physical qubits:    114,973
runtime:             8.30 hrs
total error:         0.21555
─────────────────────────────
code distance:       13 (|α|² = 20.00)
#factories:          59
factories distance:  9 (|α|² = 12.83)
factory fraction:    4.36%
─────────────────────────────



The cool thing about Qualtran is that with this interface we can easily perform a resource estimator for the large number of algorithms that are contained in the Qualtran library. Here is an example for the QFT on a register of 100 qubits.

In [13]:
from qualtran.bloqs.qft import QFTTextBook

qft_text_book = QFTTextBook(100)


result_qft = estimate_from_qualtran(
    qft_text_book,
    frontier=False,
    error_total=None,
    error_budget=(0.333 * 0.5, 0.333 * 0.5, 0.0),
)

print(result_qft)

Parameters obtained from the Rust resource estimator
─────────────────────────────
# physical qubits:    3,205
runtime:             0.00 hrs
total error:         0.11650
─────────────────────────────
code distance:       7 (|α|² = 11.00)
#factories:          4
factories distance:  5 (|α|² = 7.15)
factory fraction:    5.62%
─────────────────────────────



## Estimation from a Q# file
The last option is to specify an algorithm in a Q# file and pass it as an input to the resource estimator. The docstring can be seen below.

In [14]:
print(qre.estimate_file_struct.__doc__)  # type: ignore

Estimate resources from a Q# file and return both the best estimate and, optionally,
a frontier of Pareto-optimal trade-offs, together with the parsed logical counts.

# Arguments
- `filename` — Path to a Q# source file to be parsed and interpreted for counts.
- `frontier` — If `true`, also compute a frontier of estimates (e.g., different distances/α).
- `error_total` — Overall error target `p_total`; mutually exclusive with `error_budget`.
- `error_budget` — Tuple `(target, meas, routing)` if an explicit split is desired.

# Returns
A 3-tuple:
1. `EstimatesPy` — the single best estimate,
2. `Vec<EstimatesPy>` — optionally, the frontier (empty if `frontier == false`),
3. `LogicalCountsPy` — Python snapshot of the logical counts extracted from `filename`.

# Errors
- I/O or parsing failures when loading the Q# file,
- Failures during resource estimation.



To perform the optimization we simply have to provide the path to the Q# file we are interested in. An example is given in ``qsharp/Adder.rs``.

In [15]:
filename = "qsharp/Adder.qs"
single_qsharp, frontier, counts = qre.estimate_file_struct(  # type: ignore
    filename, frontier=True, error_total=None, error_budget=(0.0005, 0.0005, 0.0)
)
print("\n=== Q# example ===")
print(single_qsharp)


=== Q# example ===
Parameters obtained from the Rust resource estimator
─────────────────────────────
# physical qubits:    13,419
runtime:             0.00 hrs
total error:         0.00019
─────────────────────────────
code distance:       9 (|α|² = 14.00)
#factories:          2
factories distance:  5 (|α|² = 8.18)
factory fraction:    0.67%
─────────────────────────────

